Extending 04 notebook to find the loss and back propagate the gradients to minimize loss and update the parameters (weights & biases). 

10 5D inputs, hidden layer of 4 neurons, 1 output.

In [89]:
import torch
import torch.nn as nn

In [90]:
input = torch.rand(10, 5, dtype=torch.float32)
print(f"input: {input}")
output = torch.tensor([int(sum(i)*10) for i in input], dtype=torch.float32)
print(f"output: {output}")
neurons = 4

input: tensor([[0.4790, 0.4153, 0.4578, 0.0200, 0.8116],
        [0.3253, 0.3568, 0.3292, 0.2158, 0.0364],
        [0.2391, 0.3410, 0.5000, 0.5632, 0.4284],
        [0.9182, 0.3176, 0.4751, 0.7543, 0.4853],
        [0.2307, 0.0407, 0.5885, 0.9183, 0.4529],
        [0.2639, 0.2506, 0.3829, 0.6389, 0.0396],
        [0.5757, 0.8574, 0.0241, 0.9974, 0.1817],
        [0.8144, 0.3778, 0.2622, 0.8727, 0.4420],
        [0.0330, 0.0425, 0.5193, 0.9776, 0.7235],
        [0.9493, 0.8257, 0.2241, 0.0965, 0.6537]])
output: tensor([21., 12., 20., 29., 22., 15., 26., 27., 22., 27.])


In [91]:
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, x):
        return self.layers(x)

In [92]:
# model initialization
model = MultiLayerPerceptron(input_size=input.shape[-1], output_size=output.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

layers.0.weight: tensor([[-0.2984, -0.2067,  0.2010, -0.2228, -0.3532],
        [-0.4013,  0.3781, -0.2861, -0.2335,  0.1375],
        [ 0.3646,  0.0632,  0.3983, -0.0092,  0.3889],
        [-0.0354, -0.2857, -0.3245, -0.1860, -0.1735]])
layers.0.bias: tensor([ 0.2094,  0.0729, -0.1472, -0.4096])
layers.2.weight: tensor([[-0.2918,  0.1674, -0.1340, -0.2943],
        [-0.0616, -0.0267,  0.4136, -0.2275],
        [ 0.2649, -0.0269,  0.1843, -0.4328],
        [-0.0080,  0.2609, -0.3087, -0.0917],
        [-0.2490, -0.0642, -0.1567, -0.4285],
        [ 0.2413, -0.2728,  0.2053,  0.0734],
        [ 0.1807, -0.1036, -0.1420,  0.3420],
        [ 0.3438, -0.1903,  0.2881, -0.3996],
        [ 0.4158, -0.1181,  0.3887, -0.1218],
        [-0.3688, -0.2396,  0.3851,  0.0935]])
layers.2.bias: tensor([ 0.0773, -0.4488,  0.0445, -0.4296,  0.2738, -0.4886, -0.1895,  0.4292,
        -0.2749, -0.0883])


In [93]:
epochs = 100
for _ in range(epochs):
    # forward pass
    z = model(input)
    # calculate loss
    loss = nn.MSELoss()(z, output)
    print(f"loss: {loss.item()}")
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    for name, param in model.named_parameters():
        print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

loss: 517.6592407226562
layers.0.weight.grad: tensor([[-0.0623, -0.0683, -0.0631, -0.0413, -0.0070],
        [ 0.3430,  0.2980,  0.1638,  0.0280,  0.3520],
        [-1.8737, -1.4874, -1.4633, -2.3585, -1.6501],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
layers.0.bias.grad: tensor([-0.1915,  0.4804, -3.8908,  0.0000])
layers.2.weight.grad: tensor([[-0.0184, -0.0089, -1.5224,  0.0000],
        [-0.0109, -0.0052, -0.8901,  0.0000],
        [-0.0175, -0.0084, -1.4420,  0.0000],
        [-0.0258, -0.0126, -2.1452,  0.0000],
        [-0.0191, -0.0093, -1.5815,  0.0000],
        [-0.0135, -0.0065, -1.1173,  0.0000],
        [-0.0230, -0.0111, -1.9048,  0.0000],
        [-0.0233, -0.0112, -1.9187,  0.0000],
        [-0.0195, -0.0094, -1.6038,  0.0000],
        [-0.0237, -0.0114, -1.9533,  0.0000]])
layers.2.bias.grad: tensor([-4.1944, -2.4598, -3.9775, -5.9082, -4.3569, -3.0827, -5.2481, -5.2930,
        -4.4265, -5.3901])
loss: 516.8953247070312
layers.0.weight.grad: tensor([[-0.

In [94]:
# prediction
output_pred = model(input)
print(f"output_pred: {output_pred.data}")

output_pred: tensor([[10.0002, 11.4344, 13.1191,  9.9736, 10.2772, 12.1513, 11.5231, 14.5181,
         14.2798, 11.9036],
        [ 7.3350,  8.0881,  9.5678,  7.2762,  7.6030,  8.7084,  8.4397, 10.6686,
         10.2813,  8.4976],
        [ 9.5655, 10.8560, 12.5705,  9.5857,  9.8484, 11.6156, 11.0763, 13.9184,
         13.6535, 11.2819],
        [11.9471, 13.8337, 15.7558, 12.0163, 12.2410, 14.7022, 13.8535, 17.3693,
         17.2366, 14.2998],
        [10.0125, 11.4050, 13.1779, 10.0581, 10.2998, 12.2030, 11.6149, 14.5749,
         14.3341, 11.8281],
        [ 8.1242,  9.0553, 10.6418,  8.1130,  8.4003,  9.7469,  9.3937, 11.8291,
         11.4843,  9.4580],
        [10.6219, 12.1680, 13.9917, 10.6781, 10.9117, 12.9918, 12.3235, 15.4568,
         15.2499, 12.6026],
        [11.2682, 12.9912, 14.8419, 11.3134, 11.5575, 13.8174, 13.0510, 16.3801,
         16.2103, 13.4523],
        [10.1242, 11.5511, 13.3210, 10.1615, 10.4105, 12.3425, 11.7338, 14.7310,
         14.4969, 11.9829],
      